# Module 6.6.1 — Connecting Spark to Google Cloud Storage

**What you will learn:**
- How to configure Spark to read data from `gs://` (Google Cloud Storage URLs)
- How to download and use the GCS connector JAR file
- How to provide your Google credentials to Spark
- How to verify the connection by reading data from GCS

**Before you run this notebook:**
1. Upload your local parquet data to GCS (run in terminal):
   ```bash
   gsutil -m cp -r data/pq/ gs://YOUR-BUCKET-NAME/pq
   ```
2. Download the GCS connector JAR (run in terminal):
   ```bash
   mkdir -p lib
   gsutil cp gs://hadoop-lib/gcs/gcs-connector-hadoop3-2.2.5.jar ./lib/
   ```
3. Make sure your credentials JSON file is at:
   `~/.google/credentials/google_credentials.json`

**Your bucket name:** Replace `dtc_data_lake_de-zoomcamp-nytaxi` with your own bucket name.

## Step 1 — Upload Local Data to GCS (Terminal Command)

Run this in your terminal **before** running any cells below.
This uploads all your local parquet files to GCS so Spark can read them.

In [ ]:
# What happens: Shows you the terminal command to upload data.
# This cell just prints the command — you copy and run it in your terminal.
# The actual upload happens in terminal, not inside this notebook.
# -m = parallel upload (uses multiple CPU cores, much faster)
# -r = recursive (upload entire folder including subfolders)
BUCKET = 'dtc_data_lake_de-zoomcamp-nytaxi'  # CHANGE THIS to your bucket name

print(f"Run this in your terminal to upload data:")
print(f"gsutil -m cp -r data/pq/ gs://{BUCKET}/pq")

## Step 2 — Download the GCS Connector JAR (Terminal Command)

Spark does not know how to connect to Google Cloud Storage by default.
This JAR file teaches Spark how to read from `gs://` URLs.

Run this in your terminal **once** before starting the notebook.

In [ ]:
# What happens: Checks if the JAR file already exists locally.
# If it does not exist, prints the command to download it.
# The JAR file is stored in a public Google Cloud Storage bucket.
import os

jar_path = './lib/gcs-connector-hadoop3-2.2.5.jar'

if os.path.exists(jar_path):
    print(f"JAR file found at: {jar_path}")
    print("You are ready to proceed!")
else:
    print("JAR file NOT found. Run these commands in your terminal:")
    print("  mkdir -p lib")
    print("  gsutil cp gs://hadoop-lib/gcs/gcs-connector-hadoop3-2.2.5.jar ./lib/")

## Step 3 — Set Your Credentials Path

Spark needs your Google credentials to authenticate with GCS.
This is the JSON key file you downloaded when creating a service account.

In [ ]:
# What happens: Sets the path to your Google credentials JSON file.
# This file allows Spark to authenticate with Google Cloud.
# Change this path to match where your credentials file is stored.
import os

# Common credential locations — adjust to match your setup
credentials_location = os.path.expanduser('~/.google/credentials/google_credentials.json')

# Verify the file exists
if os.path.exists(credentials_location):
    print(f"Credentials found at: {credentials_location}")
else:
    print(f"WARNING: Credentials NOT found at: {credentials_location}")
    print("Update the credentials_location variable above with your actual path.")

## Step 4 — Import Required Libraries

For GCS connection we need two extra things beyond the usual SparkSession:
- `SparkConf` — holds all Spark configuration settings
- `SparkContext` — the low-level Spark engine (SparkSession wraps this)

In [ ]:
# What happens: Imports all libraries needed for the GCS connection.
# SparkConf is like a settings dictionary — we pass config to it before starting Spark.
# SparkContext is the actual Spark engine. SparkSession wraps SparkContext.
# Usually you only use SparkSession, but GCS setup requires direct SparkContext access.
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

print("PySpark version:", pyspark.__version__)
print("All imports successful.")

## Step 5 — Configure Spark for GCS Access

This is the most important step. We create a `SparkConf` object and add:
1. The path to the GCS connector JAR
2. The path to our Google credentials
3. Tell Spark to use service account authentication

This configuration must be set **before** starting Spark.

In [ ]:
# What happens: Creates a SparkConf object with all settings needed for GCS.
# .setMaster('local[*]') = still running locally, using all CPU cores
# .setAppName('test') = name shown in the Spark dashboard
# .set('spark.jars', ...) = tells Spark to load the GCS connector JAR
# The auth settings tell Spark: 'use service account + this credentials file'
# No job runs here — we are just building the configuration object.
conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

print("Spark configuration created.")
print("JAR path:", "./lib/gcs-connector-hadoop3-2.2.5.jar")
print("Credentials:", credentials_location)

## Step 6 — Create SparkContext with the Configuration

Now we create the Spark engine using our config.
We also configure the Hadoop file system settings to tell Spark:
> "When you see a URL that starts with `gs://`, use the GCS library to handle it."

In [ ]:
# What happens: Creates the SparkContext (Spark engine) with our GCS configuration.
# Then accesses the Hadoop config inside Spark to add GCS file system settings.
# hadoop_conf.set('fs.AbstractFileSystem.gs.impl', ...) means:
#   'When you see gs:// in a URL, use this Java class to handle the connection'
# hadoop_conf.set('fs.gs.impl', ...) does the same for standard file system API.
# These settings tell Spark HOW to connect to GCS (which library to use).
# Check Dashboard: After running this, open http://localhost:4040
#   You should see the Spark UI is running (no jobs yet, but engine is started).
sc = SparkContext(conf=conf)

# Access Hadoop configuration inside Spark
hadoop_conf = sc._jvm.org.apache.hadoop.conf.Configuration()

# Tell Hadoop: gs:// URLs should use the Google Cloud Storage file system
hadoop_conf.set("fs.AbstractFileSystem.gs.impl",
                "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl",
                "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")

# Tell Hadoop: use service account authentication with this credentials file
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

print("SparkContext created and Hadoop config applied.")
print("Check: http://localhost:4040 — Spark UI should be running.")

## Step 7 — Create SparkSession from the SparkContext

Normally you use `SparkSession.builder` directly.
Here we first created a `SparkContext` (Step 6), then wrap it in a `SparkSession`.
This is necessary so our GCS configuration is applied correctly.

In [ ]:
# What happens: Creates a SparkSession from the existing SparkContext.
# .config(conf=sc.getConf()) passes all our GCS settings from Step 5.
# getOrCreate() returns the existing session if one exists.
# After this, 'spark' is ready to use just like in previous notebooks
# — but now it knows how to connect to gs:// URLs.
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession ready. Spark version:", spark.version)

## Step 8 — Test the Connection: Read Data from GCS

Now the real test. We try to read parquet files directly from your GCS bucket.
If the connection works, Spark will be able to list the files and read them.

In [ ]:
# What happens: Tells Spark to read parquet files from Google Cloud Storage.
# The path starts with gs:// instead of a local path.
# Thanks to the JAR and config from Steps 5-6, Spark now understands this URL.
# This is still LAZY — Spark reads the file metadata but not the data itself yet.
# CHANGE THE BUCKET NAME below to match your own bucket.
BUCKET = 'dtc_data_lake_de-zoomcamp-nytaxi'  # <-- Change this!

df_green = spark.read.parquet(f'gs://{BUCKET}/pq/green/*/*')

print("Schema of green taxi data from GCS:")
df_green.printSchema()

## Step 9 — Preview the Data (Triggers a Spark Job)

Calling `.show()` forces Spark to actually read data from GCS.
This is the first real Spark job — watch the dashboard!

In [ ]:
# What happens: Triggers the first Spark job that reads real data from GCS.
# Spark will download a portion of the parquet files from GCS to process them.
# Check Dashboard: http://localhost:4040 -> Jobs
#   You should see a new job appear and complete in a few seconds.
#   If authentication failed, you will see an error here instead of data.
df_green.show(5)

## Step 10 — Count the Rows (Verify Full GCS Access)

`.count()` forces Spark to read all partitions across all parquet files.
This is a heavier job — it fully exercises the GCS connection.

In [ ]:
# What happens: Counts all rows in the green taxi dataset stored in GCS.
# Spark reads ALL parquet files from GCS, counts rows in each partition,
# then combines the counts — all data comes from the cloud.
# Check Dashboard: http://localhost:4040 -> Jobs -> click the count job
#   -> Stages: see how Spark reads multiple files in parallel from GCS.
row_count = df_green.count()
print(f"Total green taxi rows in GCS: {row_count:,}")
print("If you see a number here — your GCS connection is working!")

## Step 11 — Read Yellow Taxi Data from GCS

In [ ]:
# What happens: Reads all yellow taxi parquet files from GCS.
# Same as Step 8 but for yellow taxi data.
# Still lazy — no job runs until we call an action like show() or count().
df_yellow = spark.read.parquet(f'gs://{BUCKET}/pq/yellow/*/*')

print("Yellow taxi columns:", df_yellow.columns[:5], "...")
print("Row count:", df_yellow.count())

## Step 12 — Run a SQL Query on GCS Data

Once the data is loaded from GCS into a DataFrame, you can use it
exactly the same way as data from local files. SQL works the same.

In [ ]:
# What happens: Registers the GCS-backed DataFrame as a SQL table.
# Then runs a GroupBy query to count trips per borough.
# The data physically lives in GCS but Spark processes it transparently.
# Check Dashboard: http://localhost:4040 -> Jobs -> click the SQL job
#   You should see it read data from gs:// partitions.
df_green.createOrReplaceTempView('green')

result = spark.sql("""
SELECT 
    PULocationID,
    COUNT(1) AS trip_count,
    SUM(total_amount) AS total_revenue
FROM green
WHERE lpep_pickup_datetime >= '2020-01-01'
GROUP BY PULocationID
ORDER BY trip_count DESC
LIMIT 10
""")

print("Top 10 pickup zones by trip count (data read from GCS):")
result.show()

## Step 13 — Write Results Back to GCS

You can also write results back to GCS. This works the same as writing
to a local folder — just use a `gs://` path.

In [ ]:
# What happens: Writes the SQL result to a parquet folder in your GCS bucket.
# mode('overwrite') replaces any existing files at that path.
# Check Dashboard: http://localhost:4040 -> Jobs -> click the write job.
#   You will see Spark reading from one gs:// path and writing to another.
# After this runs, go to your GCS bucket and look for pq/report/ folder.
result.write.mode('overwrite').parquet(f'gs://{BUCKET}/pq/report/top_zones')

print(f"Result written to: gs://{BUCKET}/pq/report/top_zones")
print("Check your GCS bucket to verify the files were created.")

---
## Summary — What You Learned

| Concept | What it means |
|---|---|
| `gs://bucket/path` | Google Cloud Storage URL — works like a local path but in the cloud |
| GCS connector JAR | Plugin that teaches Spark how to read from `gs://` URLs |
| `SparkConf` | Settings object where you configure the JAR and credentials before starting Spark |
| `SparkContext` | The Spark engine — normally hidden inside `SparkSession` |
| `hadoop_conf` | Extra config inside Spark that maps `gs://` to the GCS file system class |
| Credentials JSON | Your Google service account key — Spark uses this to authenticate with GCS |

**Once connected, everything works the same:**
- `df.show()` reads from GCS
- `df.count()` reads from GCS
- `df.write.parquet('gs://...')` writes to GCS
- SQL queries work exactly the same

**Next step:** See `08 Running Spark in the Cloud.md` for how to:
- Create a standalone local Spark cluster (6.6.2)
- Use Dataproc for a fully managed cloud cluster (6.6.3)
- Write results directly to BigQuery (6.6.4)